# 04 — Hotspot Data Cleaning

Reads raw INPE Queimadas fire hotspot CSV files for 5 states (AM, DF, MG, RS, SP)
across 2022–2025, cleans and aggregates to daily city-level summaries, and saves
to `data/processed/hotspots_clean.parquet`.

Input: 20 CSV files in `data/raw/hotspots/`
Output: one parquet file conforming to SCHEMA.md

In [2]:
import pandas as pd
from pathlib import Path

ROOT = Path().resolve()
RAW = ROOT / "data" / "raw" / "hotspots"
PROCESSED = ROOT / "data" / "processed"

print(ROOT)
print(RAW)
print("Exists:", RAW.exists())

D:\projects\air_quality_brazil
D:\projects\air_quality_brazil\data\raw\hotspots
Exists: True


In [3]:
# List all files in raw/hotspots/ and show a quick summary
# This confirms all 20 files are present and correctly named

all_files = sorted(RAW.glob("**/*.csv"))

print(f"Total files found: {len(all_files)}\n")

for f in all_files:
    # Extract state from first 2 characters of filename
    state = f.name[:2]
    year = f.parent.name
    rows = sum(1 for _ in open(f, encoding="utf-8")) - 1  # subtract header
    print(f"{state}  {year}  {rows:>6} rows  {f.name}")

Total files found: 20

AM  2022   21217 rows  AM exportador_2025-09-24 12_37_02.952865.csv
DF  2022     251 rows  DF bdqueimadas_2022-01-01_2022-12-31.csv
MG  2022    7790 rows  MG exportador_2025-10-21 21_39_16.068456.csv
RS  2022    1635 rows  RS bdqueimadas_2022-01-01_2022-12-31.csv
SP  2022    1599 rows  SP bdqueimadas_2022-01-01_2022-12-31.csv
AM  2023   19601 rows  AM exportador_2025-09-24 12_36_33.646863.csv
DF  2023      89 rows  DF bdqueimadas_2023-01-01_2023-12-31.csv
MG  2023    6498 rows  MG exportador_2025-10-15 16_17_22.602579.csv
RS  2023    1825 rows  RS bdqueimadas_2023-01-01_2023-12-31.csv
SP  2023    1666 rows  SP exportador_2025-09-29 21_06_16.922813.csv
AM  2024   25499 rows  AM exportador_2025-09-24 12_35_51.007256.csv
DF  2024     349 rows  DF bdqueimadas_2024-01-01_2024-12-31.csv
MG  2024   11787 rows  MG exportador_2025-09-29 15_20_51.184375.csv
RS  2024    1572 rows  RS bdqueimadas_2024-01-01_2024-12-31.csv
SP  2024    8712 rows  SP exportador_2025-09-16 14_11

In [4]:
# Read one file to understand the raw format
# Using MG 2024 — largest state file with most variety in the data

sample_path = RAW / "2024" / next((RAW / "2024").glob("MG*.csv")).name

df_sample = pd.read_csv(sample_path, encoding="utf-8")

print(f"Shape: {df_sample.shape}")
print(f"\nColumns: {df_sample.columns.tolist()}")
print(f"\nDtypes:\n{df_sample.dtypes}")
print(f"\nFirst 3 rows:")
print(df_sample.head(3).to_string(index=False))
print(f"\nUnique values in RiscoFogo: {sorted(df_sample['RiscoFogo'].unique()[:10])}")

Shape: (11787, 12)

Columns: ['DataHora', 'Satelite', 'Pais', 'Estado', 'Municipio', 'Bioma', 'DiaSemChuva', 'Precipitacao', 'RiscoFogo', 'FRP', 'Latitude', 'Longitude']

Dtypes:
DataHora            str
Satelite            str
Pais                str
Estado              str
Municipio           str
Bioma               str
DiaSemChuva       int64
Precipitacao    float64
RiscoFogo       float64
FRP             float64
Latitude        float64
Longitude       float64
dtype: object

First 3 rows:
           DataHora Satelite   Pais       Estado       Municipio          Bioma  DiaSemChuva  Precipitacao  RiscoFogo  FRP  Latitude  Longitude
2024/01/04 16:49:00 AQUA_M-T Brasil MINAS GERAIS SALTO DA DIVISA Mata Atlântica            7         32.92        0.0 11.2 -15.97245  -40.12222
2024/01/06 16:34:00 AQUA_M-T Brasil MINAS GERAIS         BURITIS        Cerrado            0         11.74        0.0 38.2 -15.27273  -46.40012
2024/01/06 16:34:00 AQUA_M-T Brasil MINAS GERAIS         BURITIS        

In [5]:
def read_hotspot_file(filepath):
    """
    Reads a single INPE Queimadas hotspot CSV file.
    Returns a cleaned DataFrame with one row per hotspot detection.
    """

    df = pd.read_csv(filepath, encoding="utf-8")

    # Parse datetime and extract date only — we aggregate to daily later
    df["date"] = pd.to_datetime(
        df["DataHora"], format="%Y/%m/%d %H:%M:%S"
    ).dt.date

    # Replace -999 sentinel values with NaN across all numeric columns
    # -999 is INPE's code for missing/invalid readings
    df = df.replace(-999.0, pd.NA)

    # Drop columns that carry no analytical value
    df = df.drop(columns=["Satelite", "Pais", "DataHora"])

    # Map state names to standardized city names
    city_map = {
        "AMAZONAS": "Manaus",
        "DISTRITO FEDERAL": "Brasília",
        "MINAS GERAIS": "Belo Horizonte",
        "RIO GRANDE DO SUL": "Porto Alegre",
        "SÃO PAULO": "São Paulo",
    }
    df["city"] = df["Estado"].map(city_map)

    # Add year from parent folder name
    df["year"] = int(filepath.parent.name)

    return df


# Test on sample file
df_test = read_hotspot_file(sample_path)
print(f"Shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")
print(df_test.head(3).to_string(index=False))

Shape: (11787, 12)
Columns: ['Estado', 'Municipio', 'Bioma', 'DiaSemChuva', 'Precipitacao', 'RiscoFogo', 'FRP', 'Latitude', 'Longitude', 'date', 'city', 'year']
      Estado       Municipio          Bioma  DiaSemChuva  Precipitacao RiscoFogo  FRP  Latitude  Longitude       date           city  year
MINAS GERAIS SALTO DA DIVISA Mata Atlântica            7         32.92       0.0 11.2 -15.97245  -40.12222 2024-01-04 Belo Horizonte  2024
MINAS GERAIS         BURITIS        Cerrado            0         11.74       0.0 38.2 -15.27273  -46.40012 2024-01-06 Belo Horizonte  2024
MINAS GERAIS         BURITIS        Cerrado            0         11.85       0.0 33.4 -15.27629  -46.39568 2024-01-06 Belo Horizonte  2024


In [6]:
# Load all 20 files and concatenate into one raw DataFrame
all_files = sorted(RAW.glob("**/*.csv"))

dfs = []
for filepath in all_files:
    try:
        df = read_hotspot_file(filepath)
        dfs.append(df)
    except Exception as e:
        print(f"ERROR in {filepath.name}: {e}")

df_raw = pd.concat(dfs, ignore_index=True)

print(f"Total hotspot detections: {len(df_raw):,}")
print(f"Date range: {df_raw['date'].min()} → {df_raw['date'].max()}")
print(f"Cities: {sorted(df_raw['city'].unique())}")
print(f"Nulls in RiscoFogo: {df_raw['RiscoFogo'].isna().sum():,}")

Total hotspot detections: 127,581
Date range: 2022-01-01 → 2025-12-31
Cities: ['Belo Horizonte', 'Brasília', 'Manaus', 'Porto Alegre', 'São Paulo']
Nulls in RiscoFogo: 2,519


In [7]:
# Aggregate from individual hotspot detections to daily city-level summaries
# This produces one row per city per day — the granularity that joins
# cleanly with the INMET weather data

df_daily = (
    df_raw.groupby(["city", "date", "year"])
    .agg(
        hotspot_count=("FRP", "count"),        # number of detections that day
        frp_mean=("FRP", "mean"),              # mean fire intensity
        frp_total=("FRP", "sum"),              # total fire energy released
        risk_fire_mean=("RiscoFogo", "mean"),  # mean fire risk (excludes NaN)
        days_no_rain_mean=("DiaSemChuva", "mean"),  # mean days without rain
        precip_mean=("Precipitacao", "mean"),  # mean precipitation at hotspots
        biome_primary=("Bioma", lambda x: x.mode()[0]),  # dominant biome
    )
    .reset_index()
)

# Convert date to datetime for consistency with INMET data
df_daily["date"] = pd.to_datetime(df_daily["date"])

print(f"Shape: {df_daily.shape}")
print(f"Date range: {df_daily['date'].min()} → {df_daily['date'].max()}")
print(f"\nSample output:")
print(df_daily.head(5).to_string(index=False))

Shape: (3864, 10)
Date range: 2022-01-01 00:00:00 → 2025-12-31 00:00:00

Sample output:
          city       date  year  hotspot_count  frp_mean  frp_total risk_fire_mean days_no_rain_mean  precip_mean  biome_primary
Belo Horizonte 2022-01-01  2022              2  5.950000       11.9            0.0               1.5     4.600000 Mata Atlântica
Belo Horizonte 2022-01-02  2022              3 10.733333       32.2       0.133333          3.666667     3.533333 Mata Atlântica
Belo Horizonte 2022-01-10  2022              1  5.900000        5.9            0.0               0.0     0.100000        Cerrado
Belo Horizonte 2022-01-11  2022              1  7.200000        7.2            0.5               0.0     0.000000 Mata Atlântica
Belo Horizonte 2022-01-12  2022              7 24.785714      173.5            0.0          1.857143     0.442857 Mata Atlântica


In [8]:
# Add UTC timestamp to match INMET schema
# SCHEMA.md requires timestamp as datetime64[us, UTC]
# Hotspot data is daily so we use midnight UTC as the anchor

df_daily["timestamp"] = pd.to_datetime(df_daily["date"], utc=True)

# Reorder columns to put identifiers first
df_daily = df_daily[[
    "timestamp", "date", "city", "year",
    "hotspot_count", "frp_mean", "frp_total",
    "risk_fire_mean", "days_no_rain_mean", "precip_mean",
    "biome_primary"
]]

# Final check
print(f"Shape: {df_daily.shape}")
print(f"\nDtypes:")
print(df_daily.dtypes)
print(f"\nNulls:")
print(df_daily.isnull().sum())

Shape: (3864, 11)

Dtypes:
timestamp            datetime64[s, UTC]
date                      datetime64[s]
city                                str
year                              int64
hotspot_count                     int64
frp_mean                        float64
frp_total                       float64
risk_fire_mean                   object
days_no_rain_mean                object
precip_mean                     float64
biome_primary                       str
dtype: object

Nulls:
timestamp             0
date                  0
city                  0
year                  0
hotspot_count         0
frp_mean              0
frp_total             0
risk_fire_mean       98
days_no_rain_mean    24
precip_mean           7
biome_primary         0
dtype: int64


In [9]:
# Cast object columns to float64
# pd.NA mixed with numeric values creates object dtype after aggregation
# errors="coerce" turns any non-numeric values to NaN safely

df_daily["risk_fire_mean"] = pd.to_numeric(df_daily["risk_fire_mean"], errors="coerce")
df_daily["days_no_rain_mean"] = pd.to_numeric(df_daily["days_no_rain_mean"], errors="coerce")

# Verify dtypes are now correct
print(df_daily[["risk_fire_mean", "days_no_rain_mean"]].dtypes)
print(f"\nNulls unchanged:")
print(df_daily[["risk_fire_mean", "days_no_rain_mean"]].isnull().sum())

risk_fire_mean       float64
days_no_rain_mean    float64
dtype: object

Nulls unchanged:
risk_fire_mean       98
days_no_rain_mean    24
dtype: int64


In [10]:
# Save to processed/
# Matches SCHEMA.md requirements — parquet format, UTC timestamp, city standardized

output_path = PROCESSED / "hotspots_clean.parquet"
df_daily.to_parquet(output_path, index=False)

print(f"Saved to {output_path}")
print(f"Shape: {df_daily.shape}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Saved to D:\projects\air_quality_brazil\data\processed\hotspots_clean.parquet
Shape: (3864, 11)
File size: 0.13 MB


In [11]:
# Reload and verify everything survived the round-trip
df_check = pd.read_parquet(PROCESSED / "hotspots_clean.parquet")

print(f"Shape: {df_check.shape}")
print(f"\nDtypes:")
print(df_check.dtypes)
print(f"\nDate range: {df_check['timestamp'].min()} → {df_check['timestamp'].max()}")
print(f"\nCities: {sorted(df_check['city'].unique())}")
print(f"\nRows per city:")
print(df_check['city'].value_counts().sort_index())
print(f"\nNulls:")
print(df_check.isnull().sum())

Shape: (3864, 11)

Dtypes:
timestamp            datetime64[ms, UTC]
date                      datetime64[ms]
city                                 str
year                               int64
hotspot_count                      int64
frp_mean                         float64
frp_total                        float64
risk_fire_mean                   float64
days_no_rain_mean                float64
precip_mean                      float64
biome_primary                        str
dtype: object

Date range: 2022-01-01 00:00:00+00:00 → 2025-12-31 00:00:00+00:00

Cities: ['Belo Horizonte', 'Brasília', 'Manaus', 'Porto Alegre', 'São Paulo']

Rows per city:
city
Belo Horizonte    1167
Brasília           249
Manaus             984
Porto Alegre       645
São Paulo          819
Name: count, dtype: int64

Nulls:
timestamp             0
date                  0
city                  0
year                  0
hotspot_count         0
frp_mean              0
frp_total             0
risk_fire_mean       98


# Cleaning Summary — hotspots

## Input
- 20 CSV files from INPE Queimadas (AQUA_M-T satellite)
- 5 states: AM, DF, MG, RS, SP
- 127,581 individual hotspot detections across 2022–2025

## Transformations
- Parsed DataHora to date (daily aggregation anchor)
- Replaced -999 sentinel values with NaN
- Dropped Satelite and Pais (constant, no analytical value)
- Mapped state names to standardized city names
- Aggregated from individual detections to daily city-level summaries

## Output
- 3,864 rows × 11 columns
- Only days with ≥1 hotspot detection — zero-hotspot days absent by design
- Saved to data/processed/hotspots_clean.parquet (0.13 MB)

## Null policy
- risk_fire_mean: 98 nulls — from -999 sentinel replacement
- days_no_rain_mean: 24 nulls — from -999 sentinel replacement  
- precip_mean: 7 nulls — from -999 sentinel replacement
- All other columns: 0 nulls

## Note for integration
Zero-hotspot days will appear as NaN when joined to INMET weather data
on date + city — fill with 0 during integration, not here